# MOD-03 · NB-06 — Poisson & Negative Binomial Regression for Count Outcomes
### Health Analytics with Python · Module 03: Statistical Inference
---
**Learning objectives**
- Identify when count regression is needed (ED visits, hospital admissions, procedures)
- Fit Poisson regression and interpret incidence rate ratios (IRRs)
- Test for overdispersion and switch to negative binomial when needed
- Handle offset terms for rate modelling (visits per person-year)
- Fit zero-inflated models for excess zeros

**Estimated time:** 2 hours | **Level:** Advanced | **Libraries:** `pandas`, `numpy`, `statsmodels`, `scipy`, `matplotlib`


## 1. Setup & Count Outcome Dataset

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
from statsmodels.discrete.count_model import ZeroInflatedPoisson, ZeroInflatedNegativeBinomialP
from scipy import stats
import warnings
warnings.filterwarnings('ignore')
import os; os.makedirs('/tmp/mod03',exist_ok=True)

plt.rcParams.update({'figure.dpi':120,'figure.facecolor':'white',
                     'axes.spines.top':False,'axes.spines.right':False})

np.random.seed(42); N=1500
def logistic(x): return 1/(1+np.exp(-x))
base = np.random.normal(size=(N,3))
np.random.seed(99)
ages          = np.random.normal(58,15,N).clip(18,90).astype(int)
sexes         = np.random.choice(['M','F'],N,p=[0.48,0.52])
diabetes      = np.random.binomial(1,logistic(0.6*base[:,0]-0.5),N)
heart_failure = np.random.binomial(1,logistic(0.4*base[:,1]-1.2),N)
copd          = np.random.binomial(1,logistic(0.5*base[:,2]-1.0),N)
mental_health = np.random.binomial(1,logistic(0.3*base[:,2]-1.4),N)
payers        = np.random.choice(['Medicare','Medicaid','Private','Self-pay'],N,p=[0.38,0.24,0.30,0.08])
follow_up_yrs = np.random.uniform(0.5,3.0,N).round(2)  # Person-years of follow-up

# ED visits count: Poisson with mean depending on predictors
log_mu = (-0.3 + 0.5*diabetes + 0.8*heart_failure + 0.6*copd
          + 0.02*(ages-58) + 0.4*mental_health
          + 0.3*(payers=='Medicaid').astype(int)
          + np.log(follow_up_yrs))             # offset for follow-up time
true_mu = np.exp(log_mu)

# Overdispersed count (NB with dispersion)
ed_visits_pois = np.random.poisson(true_mu)                          # Poisson
ed_visits_nb   = stats.nbinom.rvs(n=2, p=2/(2+true_mu), size=N)    # NegBin (overdispersed)

# Zero-inflated count (many true zeros)
zero_mask      = np.random.binomial(1, 0.30, N)                     # 30% structural zeros
ed_visits_zip  = np.where(zero_mask==1, 0, ed_visits_pois)          # Zero-inflated

df = pd.DataFrame({
    'patient_id'   : [f'PT-{i:05d}' for i in range(1,N+1)],
    'age':ages,'sex':sexes,'payer':payers,
    'diabetes':diabetes,'heart_failure':heart_failure,
    'copd':copd,'mental_health':mental_health,
    'follow_up_yrs': follow_up_yrs,
    'log_follow_up': np.log(follow_up_yrs),
    'ed_visits'    : ed_visits_nb,      # primary outcome (overdispersed)
    'ed_visits_pois': ed_visits_pois,   # comparison
    'ed_visits_zip' : ed_visits_zip,    # zero-inflated comparison
})
df['age_std'] = (df['age']-df['age'].mean())/df['age'].std()

print(f"Dataset: {df.shape}")
print(f"ED visits — mean: {df.ed_visits.mean():.2f}, var: {df.ed_visits.var():.2f}")
print(f"Overdispersion ratio (var/mean): {df.ed_visits.var()/df.ed_visits.mean():.2f}")
print(f"  (ratio > 1 = overdispersed → negative binomial preferred)")
print(f"Zero proportion: {(df.ed_visits==0).mean()*100:.1f}%")


## 2. Visualising Count Distributions

In [ ]:
fig, axes = plt.subplots(1,3,figsize=(15,4))
fig.suptitle('Count outcome distributions', fontsize=13)

for ax, col, title, color in [
    (axes[0],'ed_visits_pois','Poisson (equidispersed)','#4878CF'),
    (axes[1],'ed_visits',     'Negative Binomial (overdispersed)','#D65F5F'),
    (axes[2],'ed_visits_zip', 'Zero-Inflated Poisson','#6ACC65'),
]:
    counts = df[col].value_counts().sort_index()
    ax.bar(counts.index, counts.values, color=color, alpha=0.8, edgecolor='white')
    ax.set_xlabel('ED visits (count)'); ax.set_ylabel('N patients')
    ax.set_title(f'{title} mean={df[col].mean():.1f}, var={df[col].var():.1f}')
    ax.set_xlim(-0.5, min(20, counts.index.max()))

plt.tight_layout()
plt.savefig('/tmp/mod03/count_distributions.png',bbox_inches='tight'); plt.show()


## 3. Poisson Regression with Offset

In [ ]:
# Offset = log(follow_up_yrs) → models RATE (visits per person-year)
model_pois = smf.glm(
    'ed_visits ~ age_std + C(sex) + diabetes + heart_failure + copd + mental_health '
    '+ C(payer, Treatment("Private"))',
    data=df,
    family=sm.families.Poisson(),
    offset=df['log_follow_up'],
).fit()

print(model_pois.summary())


In [ ]:
# Incidence Rate Ratios (IRR) = exp(β)
irr_df = pd.DataFrame({
    'Predictor': model_pois.params.index,
    'β'        : model_pois.params.values,
    'IRR'      : np.exp(model_pois.params.values),
    'IRR_lo'   : np.exp(model_pois.conf_int()[0].values),
    'IRR_hi'   : np.exp(model_pois.conf_int()[1].values),
    'p_value'  : model_pois.pvalues.values,
}).query("Predictor != 'Intercept'").round(3)

print("Incidence Rate Ratios (IRR) — Poisson regression:")
display(irr_df.sort_values('IRR',ascending=False))
print()
hf_irr = irr_df.loc[irr_df['Predictor']=='heart_failure','IRR'].values[0]
print(f"Interpretation of heart_failure IRR = {hf_irr:.2f}:")
print(f"  Patients with HF have {(hf_irr-1)*100:.0f}% MORE ED visits per year than those without.")


## 4. Testing for Overdispersion

In [ ]:
def overdispersion_test(model_pois, df, outcome_col):
    """
    Cameron-Trivedi auxiliary regression test for overdispersion.
    H0: Var(Y) = μ  (equidispersion — Poisson OK)
    H1: Var(Y) > μ  (overdispersion — use NB)
    """
    mu_hat = model_pois.predict()
    aux_y  = ((df[outcome_col] - mu_hat)**2 - df[outcome_col]) / mu_hat
    aux_x  = sm.add_constant(mu_hat / mu_hat)  # constant column
    aux_model = sm.OLS(aux_y, mu_hat).fit()
    alpha = aux_model.params[0]
    t_stat= aux_model.tvalues[0]
    p_val = aux_model.pvalues[0]
    return alpha, t_stat, p_val

alpha, t_stat, p_val = overdispersion_test(model_pois, df, 'ed_visits')
print("Overdispersion test (Cameron-Trivedi):")
print(f"  α̂ = {alpha:.4f}")
print(f"  t = {t_stat:.3f}, p = {p_val:.4f}")
print()
if p_val < 0.05:
    print("  ✗ Significant overdispersion detected (α > 0, p < 0.05)")
    print("    → Use Negative Binomial regression")
else:
    print("  ✓ No significant overdispersion (Poisson is appropriate)")


## 5. Negative Binomial Regression

In [ ]:
model_nb = smf.negativebinomial(
    'ed_visits ~ age_std + C(sex) + diabetes + heart_failure + copd + mental_health '
    '+ C(payer, Treatment("Private"))',
    data=df,
    offset=df['log_follow_up'],
).fit(disp=0)

print(model_nb.summary())


In [ ]:
# Compare Poisson vs NB coefficients
common = model_nb.params.index.intersection(model_pois.params.index)
irr_nb = pd.DataFrame({
    'Predictor': common,
    'IRR_NB'   : np.exp(model_nb.params[common].values),
    'IRR_Pois' : np.exp(model_pois.params[common].values),
    'p_NB'     : model_nb.pvalues[common].values,
}).query("Predictor not in ['Intercept','alpha']").round(3)

print("Poisson vs Negative Binomial — IRR comparison:")
display(irr_nb)
print(f"\nNB dispersion parameter (alpha): {np.exp(model_nb.params.get('alpha',0)):.3f}")
print(f"  (alpha >> 0 confirms overdispersion)")
print(f"\nAIC comparison:")
print(f"  Poisson AIC : {model_pois.aic:.1f}")
print(f"  NB AIC      : {model_nb.aic:.1f}")
print(f"  → Model with lower AIC is preferred")


## 6. Zero-Inflated Models

In [ ]:
# When there are too many zeros (patients who never use ED)
print("Zero-inflated ED visits (structural zeros):")
print(f"  Observed zeros : {(df.ed_visits_zip==0).sum()} ({(df.ed_visits_zip==0).mean()*100:.1f}%)")
print(f"  Poisson expected zeros: {int(np.round(len(df)*np.exp(-df.ed_visits_zip.mean())))}")

# Fit Zero-Inflated Poisson
X_vars = ['age_std','diabetes','heart_failure','copd','mental_health']
X_mat  = sm.add_constant(df[X_vars].astype(float))
offset = df['log_follow_up'].values

try:
    zip_model = ZeroInflatedPoisson(
        df['ed_visits_zip'],
        exog=X_mat,
        exog_infl=X_mat,
        offset=offset
    ).fit(method='bfgs', maxiter=300, disp=False)

    print("\nZero-Inflated Poisson — selected coefficients:")
    coef_zip = pd.DataFrame({
        'coef': zip_model.params[:len(X_mat.columns)],
        'IRR' : np.exp(zip_model.params[:len(X_mat.columns)]),
    }).round(3)
    display(coef_zip)
    print(f"\nAIC: {zip_model.aic:.1f}")
except Exception as e:
    print(f"ZIP model: {e}")
    print("(ZIP convergence can be slow — try simplifying predictors or increasing maxiter)")


## 7. Forest Plot of IRRs

In [ ]:
# Forest plot — NB model IRRs
irr_plot = irr_nb.copy()
irr_plot['IRR_lo'] = np.exp(model_nb.conf_int()[0].values[:len(irr_plot)])
irr_plot['IRR_hi'] = np.exp(model_nb.conf_int()[1].values[:len(irr_plot)])
irr_plot = irr_plot[~irr_plot['Predictor'].str.contains('Intercept|alpha')]

fig, ax = plt.subplots(figsize=(9,5))
y_pos = range(len(irr_plot)-1,-1,-1)
colors = ['#D65F5F' if r>1 and p<0.05 else '#4878CF'
          for r,p in zip(irr_plot['IRR_NB'],irr_plot['p_NB'])]

for i,((_,row),color) in enumerate(zip(irr_plot.iterrows(),colors)):
    y = list(y_pos)[i]
    ax.plot([row.IRR_lo, row.IRR_hi],[y,y],color=color,lw=2)
    ax.plot(row.IRR_NB,y,'o',color=color,ms=8,zorder=5)
    ax.text(row.IRR_hi+0.05,y,
            f"IRR={row.IRR_NB:.2f} ({row.IRR_lo:.2f}–{row.IRR_hi:.2f})",
            va='center',fontsize=9)

ax.axvline(1.0,color='black',ls='--',lw=1.2,label='IRR=1 (null)')
ax.set_yticks(list(y_pos))
ax.set_yticklabels(irr_plot['Predictor'].str.replace('C\(.*\)\[T\.','',regex=True).str.replace('\]','',regex=True))
ax.set_xlabel('Incidence Rate Ratio (IRR)', fontsize=11)
ax.set_title('Negative Binomial regression — IRR for ED visits per person-year')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig('/tmp/mod03/count_forest_plot.png',bbox_inches='tight'); plt.show()


## 8. Knowledge Check
1. The overdispersion ratio (variance/mean) is 3.2 for ED visits. Which model should you use?
2. An IRR of 1.85 for COPD means what, in plain language for a clinical report?
3. The Poisson AIC is 8420 and NB AIC is 7980. Which model fits better?
4. Why must you include an offset when modelling counts with unequal follow-up times?
5. Re-fit the NB model adding an interaction term `diabetes * heart_failure`. Does the IRR for heart_failure change substantially?


In [ ]:
# Exercise 5 — interaction term
model_nb_int = smf.negativebinomial(
    'ed_visits ~ age_std + diabetes + heart_failure + copd + mental_health '
    '+ diabetes:heart_failure',
    data=df,
    offset=df['log_follow_up'],
).fit(disp=0)
print("NB model with diabetes × heart_failure interaction:")
for name in ['heart_failure','diabetes','diabetes:heart_failure']:
    if name in model_nb_int.params.index:
        irr_val = np.exp(model_nb_int.params[name])
        p_val   = model_nb_int.pvalues[name]
        print(f"  {name:30s}: IRR={irr_val:.3f}, p={p_val:.4f}")
print(f"\nBase NB AIC : {model_nb.aic:.1f}")
print(f"Int NB AIC  : {model_nb_int.aic:.1f}")


---
**Next → NB-07: Confounding, Effect Modification & Multivariable Adjustment**